# zReg — Debug / Development Notebook

This notebook is a development and debugging companion for zReg. It loads two
`.tracklets` trajectories and one Shah CSV dataset, and provides a visualisation
helper for inspecting point clouds interactively. It is intended for developers
exploring the codebase or diagnosing registration issues, not as a polished tutorial.

## NOMPI workaround

Setting `os.environ["NOMPI"] = "1"` **before** importing zreg tells the library to
skip MPI initialisation. This is required in notebook environments where `mpi4py` is
not installed: without this flag, importing zreg triggers an MPI init error that
prevents the kernel from starting. Always run this cell first, before any zreg import.

In [ ]:
import os

os.environ["NOMPI"] = "1"

In [ ]:
DATA_PATH     = "# Replace with the path to your first .tracklets file"
DATA_PATH_2   = "# Replace with the path to your second .tracklets file"
DATA_PATH_CSV = "# Replace with the path to your Shah CSV file"

# # from mpi4py import MPI
import zreg
import time
import torch

%load_ext autoreload
%autoreload 2

## Load trajectory data

Two `.tracklets` datasets and one Shah CSV are loaded here. `pcs` and `pcs2` are
time-series from two different embryo recordings; `pcs3` is loaded from a CSV file
in the Shah et al. format. If you only have some of these files, comment out the
lines you do not need — the remaining cells can be adapted accordingly.

In [ ]:
pcs, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH,
    device="cpu",
)
pcs2, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH_2,
    device="cpu",
)
pcs3 = zreg.dataset.load_shah_from_csv(
    filepath=DATA_PATH_CSV,
    device="cpu",
)

## Visualisation helper

`draw_registration_result` accepts Open3D `PointCloud` objects (not zreg dicts) and
renders them in an interactive Plotly viewer. The `target` argument is optional: if
omitted, only the source is displayed, which is useful for inspecting a single frame
or a transformed result in isolation.

In [ ]:
import copy
import open3d as o3d


def draw_registration_result(source, target=None):
    source_temp = copy.deepcopy(source)
    source_temp.paint_uniform_color([1, 0.706, 0])
    toplot = [
        source_temp.to_legacy(),
    ]
    if target is not None:
        target_temp = copy.deepcopy(target)
        target_temp.paint_uniform_color([0, 0.651, 0.929])
        toplot.append(target_temp.to_legacy())

    o3d.visualization.draw_plotly(
        toplot,
        zoom=1.2,
        # front=[0.6452, -0.3036, -0.7011],
        # lookat=[1.9892, 2.0208, 1.8945],
        # up=[-0.2779, -0.9482, 0.1556]
    )